In [ ]:
#Mount to google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Link to folder
import os
os.chdir("/content/drive/My Drive/Baby detection")

In [ ]:
!ls

 Angry	'baby emotion detector .ipynb'	'confusion matrixc66n.png'   Happy   Sad


In [ ]:
data = []
label = []

In [ ]:
#Load file
import glob
Happy = glob.glob('Happy/*.jpg') #0
Sad = glob.glob('Sad/*.jpg') #1
Angry = glob.glob('Angry/*.jpg') #2

In [ ]:
data = list(data)
label = list(label)

In [ ]:
# import numpy as np
# data1 = np.array(data)

In [ ]:
import tensorflow as tf
import numpy as np
for i in Happy:
  image = tf.keras.preprocessing.image.load_img(i,target_size=(100,120))
  image = np.array(image)
  data.append(image)
  label.append(0)

In [ ]:
def count():
  a = 0
  for i in Sad:
    a =  a + 1
  print (a)
print (count())

148
None


In [ ]:
def count():
  a = 0
  for i in Happy:
    a =  a + 1
  print (a)
print (count())

174
None


In [ ]:
def count():
  a = 0
  for i in Angry:
    a =  a + 1
  print (a)
print (count())

71
None


In [ ]:
data1 = np.array(data)
label1=np.array(label)
data1.shape

(174, 100, 120, 3)

In [ ]:
label1.shape

(174,)

In [ ]:

import tensorflow as tf
import numpy as np
for i in Angry:
  image = tf.keras.preprocessing.image.load_img(i,target_size=(100,120))
  image = np.array(image)
  data.append(image)
  label.append(2)

In [ ]:
import tensorflow as tf
import numpy as np
for i in Sad:
  image = tf.keras.preprocessing.image.load_img(i,target_size=(100,120))
  image = np.array(image)
  data.append(image)
  label.append(1)

In [ ]:
#Split data to train
data = np.array(data)
label = np.array(label)
from sklearn.model_selection import train_test_split
X_train, X_test, ytrain, ytest = train_test_split(data, label, test_size=0.2,
                                                 random_state=42)

In [ ]:
data.shape

(393, 100, 120, 3)

In [ ]:
X_train.shape

(314, 100, 120, 3)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=20,    # Xoay ảnh ngẫu nhiên
    width_shift_range=0.2, # Dịch chuyển ngang
    height_shift_range=0.2, # Dịch chuyển dọc
    horizontal_flip=True,  # Lật ảnh theo chiều ngang
    fill_mode='nearest')

datagen.fit(X_train)

In [ ]:
# =============================Run model=======================================
# =============================Run model=======================================
#tao moi truong va input thu vien de tao model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras import backend as K
import tensorflow as tf
input_shape = (100, 120, 3)
output_shape = 3
# ================== 1. Tối ưu dữ liệu đầu vào ==================
# Chuẩn hóa ảnh về [0,1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

ytrain = ytrain.astype(np.int32)
ytest = ytest.astype(np.int32)

# Kiểm tra kích thước ảnh có đồng nhất không
for i, img in enumerate(X_train):
    if img.shape != (100, 120, 3):
        print(f"Lỗi: Ảnh {i} có shape {img.shape}")


loss_function = 'sparse_categorical_crossentropy'
#optimizer_function = 'adam'#
# optimizer_function = Adam(lr=0.0002, beta_1=0.5)
optimizer_function = SGD(learning_rate=0.01, momentum=0.9)

# define cnn model
def define_CNN2D_model(in_shape, out_shape):
    convnet = Sequential()

#     convnet.add(Conv2D(filters=2, kernel_size=(3,3), strides=1, padding='same', activation='relu', input_shape=input_shape, name='conv2d_1'
#                       ,kernel_initializer='random_uniform'))

 #   convnet.add(Conv2D(filters=32, kernel_size=(3,3), strides=1, padding='same', activation='relu', input_shape=input_shape, name='conv2d_1'))


 #   convnet.add(BatchNormalization())
 #   convnet.add(Dropout(0.3))

#     convnet.add(Conv2D(filters=16, kernel_size=(5,5), strides=1, padding='same', activation='relu', name='conv2d_2'
#                       ,kernel_initializer='random_uniform'))
#     convnet.add(BatchNormalization())
#     convnet.add(Dropout(0.5))


#    convnet.add(Flatten(name='Flatten_1'))
 #   convnet.add(Dense(output_shape, activation='softmax', name='Dense_2'))


    # compile model
   # convnet.compile(optimizer=optimizer_function, loss='sparse_categorical_crossentropy', metrics=['accuracy'])# compile model
    #convnet.compile(optimizer=optimizer_function, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    convnet.add(Conv2D(32, (3,3), activation='relu', input_shape=in_shape))
    convnet.add(BatchNormalization())  # Thêm BatchNorm
    convnet.add(MaxPooling2D((2,2)))
    convnet.add(Dropout(0.3))  # Giảm Dropout để giữ lại thông tin

    convnet.add(Flatten())
    convnet.add(Dense(64, activation='relu'))
    convnet.add(BatchNormalization())  # Thêm BatchNorm ở lớp Dense
    convnet.add(Dropout(0.3))
    convnet.add(Dense(out_shape, activation='softmax'))
    # Learning Rate Decay
    initial_learning_rate = 0.001
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate, decay_steps=1000, decay_rate=0.9, staircase=True
)
    optimizer = Adam(learning_rate=lr_schedule)


    convnet.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return convnet
    # compile model

In [ ]:
cnn_model = define_CNN2D_model(input_shape, output_shape)
cnn_model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 98, 118, 32)         │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_6                │ (None, 98, 118, 32)         │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 49, 59, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 49, 59, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 92512)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │       5,920,832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_7                │ (None, 64)                  │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             195 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,922,307 (22.59 MB)

 Trainable params: 5,922,115 (22.59 MB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
# ================== 3. Giảm thời gian huấn luyện ==================
# Mixed Precision
print("\nEnabling Mixed Precision for faster training...")
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

# Bật XLA để tối ưu tốc độ tính toán
print("\nEnabling XLA Optimization...")
tf.config.optimizer.set_jit(True)

# Early Stopping & Model Checkpoint
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True)



Enabling Mixed Precision for faster training...

Enabling XLA Optimization...


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data Augmentation
print("\nApplying Data Augmentation...")
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
datagen.fit(X_train)


Applying Data Augmentation...


In [ ]:
print (X_train.shape)

(314, 100, 120, 3)


In [ ]:
import tensorflow as tf
tf.config.run_functions_eagerly(True)  # Kích hoạt eager execution

In [ ]:
# Bật mixed precision để tăng tốc với GPU
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

# Bật XLA để tối ưu tốc độ training
tf.config.optimizer.set_jit(True)

# Dùng Early Stopping và Model Checkpoint
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True)


In [ ]:
#=============================Training model==================================
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

train_model = cnn_model.fit(
    datagen.flow(X_train, ytrain, batch_size=16),
    epochs=20,
    validation_data=(X_test, ytest),
    verbose=1,


)

Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/usr/local/lib/python3.11/dist-packages/tensorflow/python/data/ops/structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


20/20 ━━━━━━━━━━━━━━━━━━━━ 55s 3s/step - accuracy: 0.3089 - loss: 1.6811 - val_accuracy: 0.3038 - val_loss: 3.2234
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 56s 3s/step - accuracy: 0.2766 - loss: 1.5622 - val_accuracy: 0.2658 - val_loss: 2.2193
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 80s 3s/step - accuracy: 0.3981 - loss: 1.2433 - val_accuracy: 0.3038 - val_loss: 1.5038
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 82s 3s/step - accuracy: 0.3646 - loss: 1.3159 - val_accuracy: 0.3038 - val_loss: 1.2674
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 83s 3s/step - accuracy: 0.3276 - loss: 1.3999 - val_accuracy: 0.5949 - val_loss: 0.9740
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 54s 3s/step - accuracy: 0.3066 - loss: 1.3242 - val_accuracy: 0.1899 - val_loss: 1.3403
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 81s 3s/step - accuracy: 0.4080 - loss: 1.3586 - val_accuracy: 0.3038 - val_loss: 1.1977
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 83s 3s/step - accuracy: 0.3614 - loss: 1.2263 - val_accuracy: 0.3038 - val_loss: 1.5679
Epo

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from imblearn.metrics import classification_report_imbalanced
import seaborn as sns
from sklearn.metrics import f1_score,accuracy_score,confusion_matrix
import itertools

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm

def confusion_plot(pred, y_true):
    sns.set(rc={'figure.figsize':(5,4)})
    fault_labels = np.unique(y_true)
    print(fault_labels)
    cm_array = confusion_matrix(y_true, pred,labels=fault_labels)
    df_cm = pd.DataFrame(cm_array, index = fault_labels,
                      columns = fault_labels)
    sns.heatmap(df_cm,annot=True)
    plt.show()

    print(classification_report_imbalanced(np.array(y_true), np.array(pred)))
    return plt

def plot_confusion_matrix(cm, classes=None,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):

    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    mpl.rcParams.update(mpl.rcParamsDefault)
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)
    plt.figure(figsize=(10, 10))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar(shrink=0.7)
    tick_marks = np.arange(len(list(range(cm.shape[0]))))
#     plt.xticks(tick_marks, classes, rotation=45)
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes,rotation=90)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    return plt


def plot_with_labels(data):
    #loop through labels and plot each cluster
    sns.set(rc={'figure.figsize':(5,5)})
    plt.figure()
    for i, label in enumerate(range(10)):

        #add data points
        plt.scatter(x=data.loc[data['label']==label, 'x'],
                    y=data.loc[data['label']==label,'y'],
                    color=cm.rainbow(int(255 * i / 9)),
                    alpha=0.20)

        #add label
        plt.annotate(label,
                     data.loc[data['label']==label,['x','y']].mean(),
                     horizontalalignment='center',
                     verticalalignment='center',
                     size=14,
                     weight='bold',
                     color='black')

In [ ]:
pred = np.argmax(cnn_model.predict(X_test), axis=1)
# utils.confusion_plot(pred,data.y_test)
plot_confusion_matrix(confusion_matrix(ytest,pred),  normalize=True,title=None)
plt.savefig("confusion matrixc66n.png")


3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 205ms/step
Normalized confusion matrix
[[0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]]
